# Nominally significant univariate results

Read and filter the univariate Cox results for the reported
`icd_or_vte_allow_other_primaries_arpi` cohort. The notebook mirrors Figure 3's
input selection: it prefers the shared-canonical-lab result file containing all
landmarks and falls back to the separate legacy landmark files when needed.

**Nominal significance means `p_value < 0.05`; it does not imply FDR significance.**

## Configuration

In [ ]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

NEPC_PROJ_PATH = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS")
COHORT = "icd_or_vte_allow_other_primaries_arpi"
NOMINAL_P_THRESHOLD = 0.05

RUN_DIR = NEPC_PROJ_PATH / "survival_analysis" / f"local_runs_{COHORT}"
RESULT_FILENAME = "cox_agg_univariate_nobs_adjusted.csv"
SHARED_RESULT = RUN_DIR / "cox" / "landmark_shared" / RESULT_FILENAME

print(f"Cohort: {COHORT}")
print(f"Run directory: {RUN_DIR}")

## Load all univariate results

In [ ]:
def load_univariate_results():
    if SHARED_RESULT.exists():
        paths = [SHARED_RESULT]
        source_type = "shared canonical labs"
    else:
        paths = sorted(RUN_DIR.glob(f"cox/landmark_*/both/{RESULT_FILENAME}"))
        source_type = "legacy per-landmark"

    if not paths:
        raise FileNotFoundError(
            f"No {RESULT_FILENAME} files found under {RUN_DIR}. "
            "Run the univariate models first."
        )

    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        if "p_value" not in frame.columns:
            raise ValueError(f"{path} does not contain a p_value column")
        if "landmark_days" not in frame.columns:
            match = re.search(r"landmark_(\\d+)", str(path))
            if match is None:
                raise ValueError(f"Could not determine landmark for {path}")
            frame.insert(0, "landmark_days", int(match.group(1)))
        frame.insert(0, "cohort", COHORT)
        frame["result_file"] = str(path)
        frames.append(frame)

    results = pd.concat(frames, ignore_index=True)
    results["p_value"] = pd.to_numeric(results["p_value"], errors="coerce")
    print(f"Loaded {len(results):,} rows from {len(paths)} {source_type} file(s):")
    for path in paths:
        print(f"  {path}")
    return results


all_univariate_results = load_univariate_results()
display(all_univariate_results.head())

## Filter to nominal significance

In [ ]:
nominal_hits = (
    all_univariate_results.loc[
        all_univariate_results["p_value"].notna()
        & (all_univariate_results["p_value"] < NOMINAL_P_THRESHOLD)
    ]
    .copy()
)

sort_columns = [
    column
    for column in ("endpoint", "landmark_days", "p_value", "feature")
    if column in nominal_hits.columns
]
nominal_hits = nominal_hits.sort_values(sort_columns).reset_index(drop=True)

group_columns = [
    column for column in ("endpoint", "landmark_days")
    if column in all_univariate_results.columns
]
if group_columns:
    tested = (
        all_univariate_results.groupby(group_columns, dropna=False)
        .size()
        .rename("n_tested")
    )
    significant = (
        nominal_hits.groupby(group_columns, dropna=False)
        .size()
        .rename("n_nominal_p_lt_0_05")
    )
    summary = pd.concat([tested, significant], axis=1).fillna(0).astype(int).reset_index()
else:
    summary = pd.DataFrame({
        "n_tested": [len(all_univariate_results)],
        "n_nominal_p_lt_0_05": [len(nominal_hits)],
    })

display(summary)
print(f"Nominally significant rows: {len(nominal_hits):,}")
display(nominal_hits)

## Optional export

Set `WRITE_CSV = True` to save the filtered table beside the model outputs.

In [ ]:
WRITE_CSV = False
OUTPUT_CSV = RUN_DIR / "cox" / "nominally_significant_univariate_results.csv"

if WRITE_CSV:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    nominal_hits.to_csv(OUTPUT_CSV, index=False)
    print(f"Wrote {len(nominal_hits):,} rows to {OUTPUT_CSV}")
else:
    print("CSV export disabled; set WRITE_CSV = True to write the filtered results.")